# Phase 1C: timestamp-corrected catalog preprocessing (v2)

Same ML target definition, previous 20 retained events and ten-feature schema. All integer timestamp arithmetic explicitly uses nanoseconds; seconds/days conversions use named constants. Independent datetime and rolling-window checks supplement prefix invariance. Outputs: dataset/v2_timestamp_fixed/. Previous datasets are preserved.

No training, tuning, waveforms or fusion. Operational catalog availability remains UNKNOWN. Research scope remains the catalog sliding-window/causal historical-summary adaptation documented in the earlier report; no exact reproduction is claimed.


In [1]:
from pathlib import Path
import json, hashlib, math, io, platform
import numpy as np
import pandas as pd
import torch
import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

ROOT=Path.cwd().resolve()
if ROOT.name!="bilstm new updated": ROOT=ROOT/"bilstm new updated"
SOURCE=ROOT.parent/"dataset_2.csv"
ORIGINAL=ROOT/"dataset"
OUT=ORIGINAL/"v2_timestamp_fixed"; REPORTS=OUT/"reports"
OUT.mkdir(parents=True,exist_ok=True); REPORTS.mkdir(exist_ok=True)
SEED=20261109
np.random.seed(SEED); torch.manual_seed(SEED)
CONFIG={"pipeline_version":"catalog_seq20_v2_timestamp_fixed","timestamp_integer_unit":"ns","sequence_length":20,"rolling_events":20,
        "rate_days":30,"train_fraction":0.70,"val_fraction":0.15,
        "magnitude_column":"Magnitude_Value","magnitude_type_column":"Magnitude_Type",
        "retained_magnitude_type":"ML","class_threshold":4.5,"seed":SEED,
        "scaler":"StandardScaler","target_transform":"identity","augmentation_enabled":False}
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""):h.update(block)
    return h.hexdigest()
def write_json(name,obj):
    p=Path(name); tmp=p.with_suffix(p.suffix+".tmp")
    tmp.write_text(json.dumps(obj,indent=2,allow_nan=False),encoding="utf-8")
    tmp.replace(p)
# Normalize datetime resolution BEFORE any integer arithmetic.
NS_PER_SECOND=1_000_000_000
SECONDS_PER_DAY=86_400
WINDOW_NS=30*SECONDS_PER_DAY*NS_PER_SECOND
def utc_nanoseconds(series):
    return pd.to_datetime(series,utc=True).dt.as_unit("ns").astype("int64").to_numpy()
def past_30d_counts(series):
    values=utc_nanoseconds(series)
    starts=np.searchsorted(values,values-WINDOW_NS,side="right")
    return np.arange(len(values))-starts+1
unit_results=[]
def unit_check(name,condition):
    assert bool(condition),name
    unit_results.append({"test":name,"status":"PASS"})
pairs=[
 ("2024-01-01T00:00:00Z","2024-01-01T00:00:01Z",1.0),
 ("2024-01-01T00:00:00Z","2024-01-01T00:00:00.250000Z",0.25),
 ("2024-01-01T00:00:00Z","2024-01-01T00:00:00.000001Z",0.000001),
 ("2024-01-01T00:00:00Z","2024-01-02T00:00:00Z",86400.0),
 ("2024-02-28T00:00:00Z","2024-03-01T00:00:00Z",172800.0),
 ("2024-01-01T00:00:00+05:30","2023-12-31T18:30:00Z",0.0)]
from datetime import datetime
for k,(a,b,expected) in enumerate(pairs):
    ref=(datetime.fromisoformat(b.replace("Z","+00:00"))-datetime.fromisoformat(a.replace("Z","+00:00"))).total_seconds()
    for native in ("us","ns"):
        times=pd.Series(pd.to_datetime([a,b],format="mixed",utc=True)).dt.as_unit(native)
        values=utc_nanoseconds(times)
        unit_check(f"known_seconds_pair_{k}_{native}",
                   np.isclose((values[1]-values[0])/NS_PER_SECOND,expected,rtol=0,atol=1e-12)
                   and ref==expected)
        unit_check(f"known_days_pair_{k}_{native}",
                   np.isclose(times.diff().dt.total_seconds().iloc[1]/SECONDS_PER_DAY,
                              expected/SECONDS_PER_DAY,rtol=0,atol=1e-15))
base=pd.Timestamp("2024-01-01",tz="UTC")
boundary=pd.Series([base,base+pd.Timedelta(1,"ns"),base+pd.Timedelta(WINDOW_NS-1,"ns"),
                    base+pd.Timedelta(WINDOW_NS,"ns"),base+pd.Timedelta(WINDOW_NS+1,"ns")])
unit_check("one_nanosecond_delta",(utc_nanoseconds(boundary)[1]-utc_nanoseconds(boundary)[0])/NS_PER_SECOND==1e-9)
unit_check("30day_boundary_expected_counts",np.array_equal(past_30d_counts(boundary),[1,2,3,3,3]))
reference_counts=pd.Series(np.ones(len(boundary)),index=pd.DatetimeIndex(boundary)).rolling("30D",closed="right").sum().to_numpy()
unit_check("30day_boundary_independent_rolling",np.array_equal(past_30d_counts(boundary),reference_counts))
unit_check("boundary_prefix_independence",np.array_equal(past_30d_counts(boundary.iloc[:3]),past_30d_counts(boundary)[:3]))
sparse=pd.Series(pd.to_datetime(["1906-08-31T14:57:30Z","1908-12-12T12:54:09Z"],utc=True))
unit_check("multi_year_gap_not_in_30day_window",np.array_equal(past_30d_counts(sparse),[1,1]))
write_json(OUT/"timestamp_unit_tests.json",{"timestamp_integer_unit":"ns","seconds_per_day":SECONDS_PER_DAY,"tests":unit_results})
original_hash_registry=json.loads((ORIGINAL/"catalog_artifact_hashes.json").read_text())
original_artifacts_before={name:sha256(ORIGINAL/name) for name in original_hash_registry["artifacts"]}
assert original_artifacts_before==original_hash_registry["artifacts"]
original_registry_hash=sha256(ORIGINAL/"catalog_artifact_hashes.json")

source_hash=sha256(SOURCE)
RUN_ID=hashlib.sha256((source_hash+json.dumps(CONFIG,sort_keys=True)).encode()).hexdigest()[:16]
existing=OUT/"preprocessing_config.json"
if existing.exists():
    prior=json.loads(existing.read_text())
    assert prior.get("run_id")==RUN_ID, "Existing catalog configuration differs; use an explicit new artifact version."
# One and only original CSV parse; preserve original text values and timestamps.
original=pd.read_csv(SOURCE,dtype=str,keep_default_na=False)
assert len(original)>20
required=["source_id","origin_time_utc","Magnitude_Value","Magnitude_Type",
          "source_magnitude","source_magnitude_type","Depth_km","Lat","Long"]
assert set(required)<=set(original.columns)
numeric=original[["Magnitude_Value","source_magnitude","Depth_km","Lat","Long"]].apply(pd.to_numeric,errors="coerce")
timestamps=pd.to_datetime(original.origin_time_utc.str.replace(" UTC","",regex=False),
                          format="mixed",utc=True,errors="coerce")
types=original.Magnitude_Type.str.strip().str.upper()
source_types=original.source_magnitude_type.str.strip().str.upper()
mag_diff=~np.isclose(numeric.Magnitude_Value,numeric.source_magnitude,atol=1e-6,rtol=0,equal_nan=True)
type_diff=types.ne(source_types)
source_missing=original.apply(lambda c:c.str.strip().eq("").sum()).astype(int).to_dict()
exact_duplicate=original.duplicated(keep=False)
id_duplicate=original.source_id.duplicated(keep=False)
physical_duplicate=pd.DataFrame({"time":timestamps,"lat":numeric.Lat,"lon":numeric.Long,
                                "depth":numeric.Depth_km,"magnitude":numeric.Magnitude_Value}).duplicated(keep=False)
inspection={"rows":len(original),"columns":len(original.columns),"column_names":original.columns.tolist(),
 "missing_by_column":source_missing,"invalid_timestamps":int(timestamps.isna().sum()),
 "duplicate_source_id_rows":int(id_duplicate.sum()),"exact_duplicate_rows":int(exact_duplicate.sum()),
 "duplicate_physical_key_rows":int(physical_duplicate.sum()),
 "tied_timestamp_rows":int(timestamps.duplicated(keep=False).sum()),
 "invalid_latitude_rows":int((~numeric.Lat.between(-90,90)).sum()),
 "invalid_longitude_rows":int((~numeric.Long.between(-180,180)).sum()),
 "nonfinite_numeric_by_column":{c:int((~np.isfinite(numeric[c])).sum()) for c in numeric},
 "numeric_ranges":{c:{"min":float(numeric[c].min()),"max":float(numeric[c].max())} for c in numeric},
 "magnitude_value_type_counts":types.value_counts().to_dict(),
 "source_magnitude_type_counts":source_types.value_counts().to_dict(),
 "magnitude_value_disagreements":int(mag_diff.sum()),"type_disagreements":int(type_diff.sum()),
 "maximum_magnitude_disagreement":float((numeric.Magnitude_Value-numeric.source_magnitude).abs().max()),
 "negative_depth_rows":int(numeric.Depth_km.lt(0).sum()),"source_sha256":source_hash}
write_json(REPORTS/"catalog_input_inspection.json",inspection)
conflicts=original.loc[mag_diff|type_diff,required].copy()
conflicts["numeric_magnitude_disagreement"]=mag_diff[mag_diff|type_diff]
conflicts["magnitude_type_disagreement"]=type_diff[mag_diff|type_diff].to_numpy()
conflicts.to_csv(OUT/"catalog_magnitude_conflicts.csv",index=False)
# Ambiguous identities/physical duplicates require source review, never a positional identity assumption.
assert not id_duplicate.any() and not exact_duplicate.any() and not physical_duplicate.any(), "Ambiguous duplicate events."
reasons=[]
for i in original.index:
    r=[]
    if not str(original.at[i,"source_id"]).strip():r.append("missing_source_id")
    if pd.isna(timestamps.iloc[i]):r.append("invalid_origin_time")
    if types.iloc[i]!="ML":r.append("non_ML_magnitude_type")
    if not np.isfinite(numeric.at[i,"Magnitude_Value"]):r.append("invalid_target_magnitude")
    if not (-90<=numeric.at[i,"Lat"]<=90):r.append("invalid_latitude")
    if not (-180<=numeric.at[i,"Long"]<=180):r.append("invalid_longitude")
    if not np.isfinite(numeric.at[i,"Depth_km"]):r.append("missing_depth")
    reasons.append(";".join(r))
disposition=pd.DataFrame({"source_id":original.source_id,"origin_time_utc_original":original.origin_time_utc,
                          "magnitude_type":types,"reason":reasons})
eligible=disposition.reason.eq("")
# Quarantine complete tied-time groups rather than inventing order among simultaneous events.
tied=eligible & timestamps[eligible].duplicated(keep=False).reindex(original.index,fill_value=False)
disposition.loc[tied,"reason"]="tied_origin_time_requires_review"
keep=disposition.reason.eq("")
disposition["retained"]=keep
disposition.to_csv(OUT/"catalog_record_disposition.csv",index=False)
catalog=pd.DataFrame({"source_id":original.loc[keep,"source_id"],
 "origin_time_utc_original":original.loc[keep,"origin_time_utc"],"time":timestamps[keep],
 "magnitude":numeric.loc[keep,"Magnitude_Value"],"magnitude_type":types[keep],
 "depth_km":numeric.loc[keep,"Depth_km"],"latitude_deg":numeric.loc[keep,"Lat"],"longitude_deg":numeric.loc[keep,"Long"]})
catalog=catalog.sort_values("time",kind="stable").reset_index(drop=True)
catalog.insert(0,"event_index",np.arange(len(catalog)))
assert catalog.source_id.is_unique and catalog.time.is_unique and catalog.time.is_monotonic_increasing
catalog.to_csv(OUT/"catalog_events.csv",index=False)
print(json.dumps({"source_rows":len(original),"retained_ML":len(catalog),"excluded":int((~keep).sum()),
                  "magnitude_disagreements":inspection["magnitude_value_disagreements"],
                  "type_disagreements":inspection["type_disagreements"]}))


{"source_rows": 8089, "retained_ML": 7125, "excluded": 964, "magnitude_disagreements": 540, "type_disagreements": 553}


## Causal feature construction

At historical event j, features use only retained ML events with origin times ≤ event j. Every input event precedes its target strictly. Rolling summaries include the observed event j and up to 19 earlier events; the rate counts events in (time(j)−30 days, time(j)]. These summaries can refer to history older than the explicit 20 input rows: dependency intervals are preserved. Initial summaries use available history only; the first inter-event gap is missing and is imputed from training inputs only. No declustering or magnitude clipping.


In [2]:
FEATURES=[
 ("historical_magnitude","ML reported units","Magnitude_Value of historical event j; never target event t"),
 ("historical_depth_km","km","Depth_km of historical event j"),
 ("historical_latitude_deg","degrees north","Lat of historical event j"),
 ("historical_longitude_deg","degrees east","Long of historical event j"),
 ("inter_event_days","days","(time(j)-time(j-1))/86400; first retained event is missing"),
 ("event_rate_30d","retained ML events/day","Count of retained events in (time(j)-30 days,time(j)] divided by 30; initial history is left-censored"),
 ("mean_magnitude_20","ML reported units","Mean magnitude over retained events max(0,j-19)..j"),
 ("std_magnitude_20","ML reported units","Population magnitude standard deviation (ddof=0) over max(0,j-19)..j; singleton=0"),
 ("mean_depth_20_km","km","Mean depth over retained events max(0,j-19)..j"),
 ("std_depth_20_km","km","Population depth standard deviation (ddof=0) over max(0,j-19)..j; singleton=0")]
names=[f[0] for f in FEATURES]
ns=utc_nanoseconds(catalog.time)
thirty_days=WINDOW_NS
left=np.searchsorted(ns,ns-thirty_days,side="right")
def build_features(cat):
    time_ns=utc_nanoseconds(cat.time)
    starts=np.searchsorted(time_ns,time_ns-thirty_days,side="right")
    mag=cat.magnitude; depth=cat.depth_km
    return pd.DataFrame({
        names[0]:mag,names[1]:depth,names[2]:cat.latitude_deg,names[3]:cat.longitude_deg,
        names[4]:cat.time.diff().dt.total_seconds()/SECONDS_PER_DAY,
        names[5]:(np.arange(len(cat))-starts+1)/30.,
        names[6]:mag.rolling(20,min_periods=1).mean(),
        names[7]:mag.rolling(20,min_periods=1).std(ddof=0),
        names[8]:depth.rolling(20,min_periods=1).mean(),
        names[9]:depth.rolling(20,min_periods=1).std(ddof=0)},columns=names)
features=build_features(catalog)
feature_array=features.to_numpy(dtype=np.float64)
# Every feature dependency is a historical catalog index no greater than its own row.
dependency_start=np.minimum.reduce([np.maximum(0,np.arange(len(catalog))-19),
                                   np.maximum(0,np.arange(len(catalog))-1),left])
feature_events=catalog[["event_index","source_id","time"]].copy()
feature_events["dependency_start_event_index"]=dependency_start
feature_events["dependency_end_event_index"]=np.arange(len(catalog))
feature_events["history_30d_left_censored"]=ns-ns[0]<thirty_days
feature_events=pd.concat([feature_events,features],axis=1)
feature_events.to_csv(OUT/"catalog_feature_events.csv",index=False)
schema={"run_id":RUN_ID,"axis_order":["samples","historical_events","features"],
 "feature_count":len(names),"sequence_length":20,"ordered_features":[
 {"index":i,"name":f[0],"units":f[1],"calculation":f[2]} for i,f in enumerate(FEATURES)],
 "operational_feature_availability":"UNKNOWN",
 "excluded":"source/trace IDs, absolute timestamps, target-event attributes, synthetic arrivals/SNR, waveform and simulator fields",
 "feature_dependency_provenance":"catalog_feature_events.csv intervals refer to catalog_events.csv event_index",
 "raw_missing_policy":"Only undefined first inter-event gap expected; train-fitted median imputation",
 "historical_scope":"Retained ML catalog only; rolling features may use older history than explicit 20-row window"}
write_json(OUT/"feature_schema.json",schema)
target_def={"run_id":RUN_ID,"primary":"Next retained ML catalog event's Magnitude_Value",
 "column":"Magnitude_Value","type_column":"Magnitude_Type","allowed_type":"ML",
 "choice_reason":"Use one explicitly recorded magnitude type; no unsupported conversion, averaging or source_magnitude substitution.",
 "alternative_columns":"source_magnitude/source_magnitude_type used solely for discrepancy audit",
 "filter_effect":"Next event means next ML-labeled event, not necessarily the next event in the unfiltered CSV.",
 "conversion":"none","source_accuracy":"UNKNOWN; ML labels and unusually large values are preserved, not independently calibrated",
 "binary_target":"1 iff raw target Magnitude_Value >= 4.5; otherwise 0",
 "target_transform":{"name":"identity","offset":0.0,"scale":1.0},
 "forecast_task":"Retrospective next-event magnitude diagnostic, not operational EEW",
 "availability":"UNKNOWN; origin times do not prove reporting availability"}
write_json(OUT/"target_definition.json",target_def)
target_idx=np.arange(20,len(catalog))
input_idx=target_idx[:,None]+np.arange(-20,0)[None,:]
N=len(target_idx); n_train=int(N*0.70); n_val=int(N*0.15)
split=np.array(["train"]*n_train+["val"]*n_val+["test"]*(N-n_train-n_val))
selections={s:np.flatnonzero(split==s) for s in ("train","val","test")}
assert (ns[input_idx]<ns[target_idx,None]).all()
assert (np.diff(ns[input_idx],axis=1)>0).all()
assert np.array_equal(input_idx[:,-1]+1,target_idx)
ids=catalog.source_id.to_numpy()
iso=catalog.time.map(lambda t:t.isoformat()).to_numpy()
original_times=catalog.origin_time_utc_original.to_numpy()
provenance=[]
for k,(t,history) in enumerate(zip(target_idx,input_idx)):
    dep=int(dependency_start[history].min())
    provenance.append({"sample_index":k,"split":split[k],"target_event_index":int(t),
        "target_event_id":ids[t],"target_timestamp_utc":iso[t],
        "target_timestamp_original":original_times[t],"target_magnitude":float(catalog.magnitude.iloc[t]),
        "target_magnitude_type":"ML","target_M_ge_4_5":int(catalog.magnitude.iloc[t]>=4.5),
        "input_event_ids":json.dumps(ids[history].tolist()),
        "input_timestamps_utc":json.dumps(iso[history].tolist()),
        "input_timestamps_original":json.dumps(original_times[history].tolist()),
        "sequence_start_timestamp":iso[history[0]],"sequence_end_timestamp":iso[history[-1]],
        "forecast_context_cutoff":iso[history[-1]],
        "input_to_target_gap_seconds":float((ns[t]-ns[history[-1]])/NS_PER_SECOND),
        "feature_dependency_start_event_index":dep,"feature_dependency_end_event_index":int(history[-1]),
        "feature_dependency_start_timestamp":iso[dep],"operational_feature_availability":"UNKNOWN"})
manifest=pd.DataFrame(provenance)
manifest.to_csv(OUT/"split_manifest.csv",index=False)
overlap={}
from itertools import combinations
for a,b in combinations(selections,2):
    ia=selections[a]; ib=selections[b]
    targets_a=set(ids[target_idx[ia]]);targets_b=set(ids[target_idx[ib]])
    inputs_a=set(ids[input_idx[ia].ravel()]);inputs_b=set(ids[input_idx[ib].ravel()])
    assert targets_a.isdisjoint(targets_b)
    overlap[a+"__"+b]={"shared_explicit_historical_events":len(inputs_a & inputs_b),
                      "earlier_targets_in_later_inputs":len(targets_a & inputs_b),
                      "later_targets_in_earlier_inputs":len(targets_b & inputs_a),
                      "shared_target_events":0,
                      "shared_explicit_history_ids":sorted(inputs_a & inputs_b)}
    assert not (targets_b & inputs_a)
write_json(OUT/"catalog_context_overlap.json",{
 "pairs":overlap,"policy":"Shared past context is permitted; target sets are disjoint.",
 "evaluation":"Sequential observed-history evaluation; earlier validation/test events can become later input history. This is not fixed-origin multi-step forecasting.",
 "dependency_scope":"Rolling feature context can extend older than the 20 explicit input events; see dependency intervals."})
print("Feature count:",len(names),"; chronological samples:",{s:len(v) for s,v in selections.items()})


Feature count: 10 ; chronological samples: {'train': 4973, 'val': 1065, 'test': 1067}


## Train-only transformation and optional augmentation

Fit median imputation and StandardScaler once on **unique events appearing in training inputs**, avoiding repeated weighting from overlapping windows. Validation/test events do not contribute to fitted parameters. Magnitude targets stay in original reported ML units (identity transform); high magnitudes and out-of-training-range values are not clipped.

Augmentation defaults **off**. The provided on-the-fly function can add seeded Gaussian noise (σ=0.01 standardized units) only to historical latitude/longitude, bounded by physical coordinate limits. Magnitude/depth and their summaries, rate, time intervals, IDs, timestamps and labels stay unchanged. This is an experimental sensitivity option, not paper-validated seismic noise.


In [3]:
train_event_idx=np.unique(input_idx[selections["train"]])
fit_values=feature_array[train_event_idx]
assert np.isfinite(fit_values[:,[0,1,2,3,5,6,7,8,9]]).all()
imputer=SimpleImputer(strategy="median")
fit_imputed=imputer.fit_transform(fit_values)
scaler=StandardScaler().fit(fit_imputed)
all_transformed=scaler.transform(imputer.transform(feature_array))
X=all_transformed[input_idx].astype(np.float32)
y=catalog.magnitude.to_numpy()[target_idx].astype(np.float32)
y_binary=(catalog.magnitude.to_numpy()[target_idx]>=4.5).astype(np.int64)
arrays={}
for s,index in selections.items():
    arrays["X_"+s]=X[index]
    arrays["y_reg_"+s]=y[index]
    arrays["y_binary_"+s]=y_binary[index]
assert all(np.isfinite(a).all() for a in arrays.values())
fit_manifest=catalog.loc[train_event_idx,["event_index","source_id","origin_time_utc_original","time"]]
fit_manifest.to_csv(OUT/"catalog_scaler_fit_events.csv",index=False)
scaler_parameters={"run_id":RUN_ID,"feature_order":names,"imputer":{"strategy":"median","statistics":imputer.statistics_.tolist()},
 "scaler":{"class":"sklearn.preprocessing.StandardScaler","mean":scaler.mean_.tolist(),
           "scale":scaler.scale_.tolist(),"var":scaler.var_.tolist(),"n_samples_seen":int(scaler.n_samples_seen_)},
 "fit_scope":"Unique catalog events present in training inputs only; no target-only or later input events",
 "fit_event_manifest":"catalog_scaler_fit_events.csv","fit_event_count":len(train_event_idx),
 "fit_latest_timestamp":iso[train_event_idx[-1]],"target_transform":{"name":"identity","offset":0.0,"scale":1.0},
 "reconstruction":"Replace NaN with imputer.statistics; X_scaled=(X_raw-scaler.mean)/scaler.scale; inverse is X_scaled*scale+mean.",
 "outlier_handling":"None; no clipping/removal of valid high magnitudes or range extremes."}
write_json(OUT/"catalog_train_scalers.json",scaler_parameters)
config={**CONFIG,"run_id":RUN_ID,"source_file":"../../../dataset_2.csv","source_sha256":source_hash,
 "feature_order":names,"retained_catalog_count":len(catalog),"original_record_count":len(original),
 "tie_policy":"Exclude all events in tied timestamp groups in the eligible ML catalog; retain disposition evidence.",
 "invalid_policy":"Exclude missing IDs/timestamps/labels/depth and invalid coordinates; stop on ambiguous duplicate identities/physical keys.",
 "fit_scope":scaler_parameters["fit_scope"],"preprocessing_parameters":"catalog_train_scalers.json",
 "target_scale":"identity; reported ML","shuffle_before_split":False,
 "partition_boundaries":{"train_samples":n_train,"val_samples":n_val,"test_samples":N-n_train-n_val},
 "read_policy":"Only original dataset_2.csv and this notebook's newly written catalog artifacts; no waveform/old dataset inputs.",
 "runtime_versions":{"python":platform.python_version(),"numpy":np.__version__,"pandas":pd.__version__,
                     "sklearn":sklearn.__version__,"torch":str(torch.__version__)},
 "operational_feature_availability":"UNKNOWN"}
write_json(OUT/"preprocessing_config.json",config)
aug_config={"run_id":RUN_ID,"enabled":False,"mode":"on-the-fly training only",
 "seed":SEED,"gaussian_std_standardized_units":0.01,"eligible_features":names[2:4],
 "eligible_feature_indices":[2,3],"physical_bounds":{"historical_latitude_deg":[-90,90],"historical_longitude_deg":[-180,180]},
 "excluded":"Magnitude/depth and rolling summaries (derived consistency); time/rate (timestamp consistency); labels, IDs, timestamps, ordering and categorical variables.",
 "implementation":"augment_training_batch in phase1_bilstm_csv_preprocessing.ipynb",
 "claim":"Experimental coordinate sensitivity augmentation; not validated measurement uncertainty or proven by cited papers."}
write_json(OUT/"augmentation_config.json",aug_config)
def augment_training_batch(x,regression,binary,*,split_name,seed=SEED,enabled=False):
    if split_name!="train":raise ValueError("Augmentation is restricted to training data.")
    x_out=np.array(x,copy=True); reg_out=np.array(regression,copy=True); bin_out=np.array(binary,copy=True)
    if enabled:
        rng=np.random.default_rng(seed)
        for col,lo,hi in [(2,-90.,90.),(3,-180.,180.)]:
            noise=rng.normal(0,aug_config["gaussian_std_standardized_units"],size=x_out.shape[:-1])
            low=(lo-scaler.mean_[col])/scaler.scale_[col]
            high=(hi-scaler.mean_[col])/scaler.scale_[col]
            x_out[...,col]=np.clip(x_out[...,col]+noise,low,high)
    return x_out,reg_out,bin_out
# Clean arrays only: augmentation is not materialized in the saved dataset.
np.savez_compressed(OUT/"bilstm_catalog_seq20.npz",**arrays)
torch.save({k:torch.from_numpy(np.ascontiguousarray(v)) for k,v in arrays.items()},OUT/"bilstm_catalog_seq20.pt")
print("Saved clean arrays; train-only fitted events:",len(train_event_idx),"; augmentation defaults off.")


Saved clean arrays; train-only fitted events: 4992 ; augmentation defaults off.


## Validation and descriptive reporting

Verify chronology, provenance, feature-prefix invariance, train-only transforms, serialized numerical equality, label preservation and source immutability. Test labels are summarized once for the requested dataset audit; no feature/model/threshold selection uses these summaries. Operational catalog availability and source magnitude calibration remain UNKNOWN.


In [4]:
checks={}
def passed(name,condition,evidence):
    assert bool(condition),name
    checks[name]={"status":"PASS","evidence":evidence}
passed("strict_input_before_target",(ns[input_idx]<ns[target_idx,None]).all(),"All 20 input origin times strictly precede each target origin time.")
passed("correct_next_event_alignment",np.array_equal(input_idx[:,-1]+1,target_idx),"Targets are immediate next events in the retained ML catalog.")
passed("chronological_target_splits",ns[target_idx[selections["train"][-1]]]<ns[target_idx[selections["val"][0]]]
       and ns[target_idx[selections["val"][-1]]]<ns[target_idx[selections["test"][0]]],"Chronological contiguous 70/15/15 target partitions; no shuffling.")
passed("unique_target_ids",len(set(ids[target_idx]))==N,"Every retained target ID occurs in exactly one sample/partition.")
# Independent prefix checks at boundaries: later events cannot alter earlier raw feature values.
prefix_tests=[]
for end in [20,int(target_idx[selections["train"][-1]]),int(target_idx[selections["val"][-1]])]:
    recomputed=build_features(catalog.iloc[:end].copy()).to_numpy()
    prefix_tests.append(np.allclose(recomputed,feature_array[:end],rtol=1e-10,atol=1e-10,equal_nan=True))
passed("past_only_feature_prefixes",all(prefix_tests),"Recomputing three catalog prefixes reproduces existing feature rows; summaries never use target/future rows.")
fit_id_set=set(ids[train_event_idx])
later_targets=set(ids[target_idx[split!="train"]])
passed("train_only_fit_ids",not(fit_id_set & later_targets) and train_event_idx[-1]<target_idx[selections["train"][-1]],
       "Saved exact unique input-event fit manifest; no validation/test target contributes.")
manual_medians=np.nanmedian(fit_values,axis=0)
manual_fit=np.where(np.isnan(fit_values),manual_medians,fit_values)
passed("train_only_fit_parameters",np.allclose(manual_medians,imputer.statistics_) and
       np.allclose(manual_fit.mean(axis=0),scaler.mean_) and np.allclose(manual_fit.var(axis=0),scaler.var_),
       "Independent training-only median/mean/variance calculation agrees with saved estimators.")
def inverse_target(z):return np.asarray(z)*target_def["target_transform"]["scale"]+target_def["target_transform"]["offset"]
passed("target_inverse_transform",np.allclose(inverse_target(y),catalog.magnitude.to_numpy()[target_idx],atol=1e-6),
       "Identity target transform; float32 values recover source target magnitudes within serialization tolerance.")
npz=np.load(OUT/"bilstm_catalog_seq20.npz",allow_pickle=False)
pt=torch.load(OUT/"bilstm_catalog_seq20.pt",map_location="cpu",weights_only=True)
passed("numerical_artifact_equality",set(npz.files)==set(pt)==set(arrays) and all(
       np.array_equal(npz[k],pt[k].numpy()) and np.array_equal(npz[k],arrays[k]) for k in arrays),
       "Every .pt tensor and .npz array exactly matches the in-memory clean array.")
passed("finite_model_arrays",all(np.isfinite(npz[k]).all() for k in npz.files),"All model input and target arrays finite.")
saved=json.loads((OUT/"catalog_train_scalers.json").read_text())
median=np.array(saved["imputer"]["statistics"]);mean=np.array(saved["scaler"]["mean"]);scale=np.array(saved["scaler"]["scale"])
reconstructed=((np.where(np.isnan(feature_array),median,feature_array)-mean)/scale)[input_idx].astype(np.float32)
passed("saved_preprocessing_reconstruction",np.array_equal(X,reconstructed),"Reloaded JSON parameters reproduce all saved X exactly.")
reloaded_manifest=pd.read_csv(OUT/"split_manifest.csv",keep_default_na=False)
provenance_ok=True
for row in reloaded_manifest.itertuples(index=False):
    k=row.sample_index; t=target_idx[k]; history=input_idx[k]
    provenance_ok &= json.loads(row.input_event_ids)==ids[history].tolist()
    provenance_ok &= json.loads(row.input_timestamps_utc)==iso[history].tolist()
    provenance_ok &= json.loads(row.input_timestamps_original)==original_times[history].tolist()
    provenance_ok &= row.target_event_id==ids[t] and row.target_timestamp_original==original_times[t]
    provenance_ok &= row.feature_dependency_start_event_index<=history[0]
    provenance_ok &= row.feature_dependency_end_event_index==history[-1]
    provenance_ok &= row.forecast_context_cutoff==iso[history[-1]]
passed("complete_event_provenance",provenance_ok,"Reloaded every row's 20 IDs/times, target identity/time, cutoff and feature dependency interval against retained source catalog.")
probe=X[selections["train"][:32]].copy()
probe_y=y[:32].copy();probe_b=y_binary[:32].copy()
a,ay,ab=augment_training_batch(probe,probe_y,probe_b,split_name="train",enabled=True)
a2,_,_=augment_training_batch(probe,probe_y,probe_b,split_name="train",enabled=True)
unchanged_cols=[i for i in range(len(names)) if i not in [2,3]]
rejects=[]
for s in ["val","test"]:
    try:augment_training_batch(probe,probe_y,probe_b,split_name=s,enabled=True)
    except ValueError:rejects.append(s)
bounds_ok=all(((a[...,col]*scale[col]+mean[col])>=lo-1e-5).all() and
             ((a[...,col]*scale[col]+mean[col])<=hi+1e-5).all()
             for col,lo,hi in [(2,-90,90),(3,-180,180)])
passed("augmentation_label_and_split_safety",np.array_equal(ay,probe_y) and np.array_equal(ab,probe_b)
       and np.array_equal(a,a2) and np.array_equal(a[...,unchanged_cols],probe[...,unchanged_cols])
       and bounds_ok and rejects==["val","test"] and np.array_equal(probe,X[:32])
       and not np.array_equal(a[...,2:4],probe[...,2:4]),
       "Optional augmentation deterministic; modifies only eligible coordinates in a copy; preserves labels; rejects validation/test.")
passed("source_csv_unchanged",sha256(SOURCE)==source_hash,"Original file SHA-256 unchanged after preprocessing.")
checks["operational_feature_availability"]={"status":"UNKNOWN","evidence":"Publication/revision times absent; origin-time causality is retrospective only."}
checks["source_magnitude_calibration"]={"status":"UNKNOWN","evidence":"ML labels are taken as reported; conflicting alternatives and unusually high values require independent catalog verification."}
checks["duplicate_events_under_different_metadata"]={"status":"UNKNOWN","evidence":"IDs/exact rows/exact physical keys checked; fuzzy duplicates and catalog completeness not independently established."}
summary={}
bins=[-np.inf,3.,4.,4.5,5.,np.inf]
labels=["<3","3–<4","4–<4.5","4.5–<5",">=5"]
for s,index in selections.items():
    raw_y=catalog.magnitude.to_numpy()[target_idx[index]]
    history_raw=feature_array[input_idx[index]]
    ranges=pd.cut(raw_y,bins=bins,labels=labels,right=False).value_counts()
    summary[s]={"samples":len(index),"X_shape":list(arrays["X_"+s].shape),
      "target_start_utc":iso[target_idx[index[0]]],"target_end_utc":iso[target_idx[index[-1]]],
      "magnitude_min":float(raw_y.min()),"magnitude_max":float(raw_y.max()),
      "magnitude_mean":float(raw_y.mean()),"magnitude_std":float(raw_y.std()),
      "magnitude_quantiles":dict(zip(["q25","median","q75"],np.quantile(raw_y,[.25,.5,.75]).tolist())),
      "magnitude_bins":{str(k):int(v) for k,v in ranges.items()},
      "M_ge_4_5_count":int((raw_y>=4.5).sum()),"M_ge_4_5_fraction":float((raw_y>=4.5).mean()),
      "raw_missing_feature_cells":int(np.isnan(history_raw).sum()),"raw_infinite_feature_cells":int(np.isinf(history_raw).sum()),
      "missing_or_invalid_model_cells":int((~np.isfinite(arrays["X_"+s])).sum()),
      "target_time_gap_days_min":float(manifest.iloc[index].input_to_target_gap_seconds.min()/SECONDS_PER_DAY),
      "target_time_gap_days_median":float(manifest.iloc[index].input_to_target_gap_seconds.median()/SECONDS_PER_DAY),
      "target_time_gap_days_max":float(manifest.iloc[index].input_to_target_gap_seconds.max()/SECONDS_PER_DAY)}
# Independent scientific-semantic checks, not a second invocation of build_features.
reference_rate=pd.Series(np.ones(len(catalog)),index=pd.DatetimeIndex(catalog.time)).rolling("30D",closed="right").sum().to_numpy()/30.
passed("independent_30day_rates_all_events",np.allclose(feature_array[:,5],reference_rate,rtol=0,atol=1e-12),
       "All retained event rates equal independent time-indexed 30D rolling counts, including left-open boundary.")
python_times=[datetime.fromisoformat(v) for v in iso]
reference_event_gaps=np.array([(python_times[i]-python_times[i-1]).total_seconds()/SECONDS_PER_DAY
                               for i in range(1,len(python_times))])
passed("independent_inter_event_days",np.allclose(feature_array[1:,4],reference_event_gaps,rtol=0,atol=1e-12),
       "All inter-event days agree with Python datetime timedelta arithmetic.")
reference_target_gaps=np.array([(python_times[t]-python_times[t-1]).total_seconds() for t in target_idx])
passed("independent_target_gap_seconds",np.allclose(manifest.input_to_target_gap_seconds,reference_target_gaps,rtol=0,atol=1e-9),
       "Every target gap matches independent datetime subtraction in seconds.")
passed("known_timestamp_pairs_and_boundaries",all(t["status"]=="PASS" for t in unit_results),
       "Seconds, sub-seconds, nanoseconds, timezone equivalence, leap day and 30-day boundary tests passed.")
old_manifest=pd.read_csv(ORIGINAL/"split_manifest.csv",keep_default_na=False)
stable_columns=[c for c in manifest.columns if c not in ["input_to_target_gap_seconds",
                 "feature_dependency_start_event_index","feature_dependency_start_timestamp"]]
passed("original_split_and_identity_preserved",
       manifest[stable_columns].astype(str).equals(old_manifest[stable_columns].astype(str)),
       "All unchanged manifest fields, including IDs, original timestamps, labels, sample order, split and cutoffs, match v1.")
old_features=pd.read_csv(ORIGINAL/"catalog_feature_events.csv")
old_raw=old_features[names].to_numpy(dtype=float)
raw_change_counts={name:int((~np.isclose(old_raw[:,j],feature_array[:,j],rtol=0,atol=1e-12,equal_nan=True)).sum())
                   for j,name in enumerate(names)}
standardized_changes={}
old_targets_match=True
with np.load(ORIGINAL/"bilstm_catalog_seq20.npz",allow_pickle=False) as old_saved:
    for split_name in selections:
        old_targets_match &= np.array_equal(old_saved["y_reg_"+split_name],arrays["y_reg_"+split_name])
        old_targets_match &= np.array_equal(old_saved["y_binary_"+split_name],arrays["y_binary_"+split_name])
        standardized_changes[split_name]={name:int((old_saved["X_"+split_name][...,j]!=arrays["X_"+split_name][...,j]).sum())
                                          for j,name in enumerate(names)}
passed("original_magnitude_targets_preserved",old_targets_match,"All regression and binary labels match v1 exactly in every partition.")
passed("expected_shapes",all(arrays["X_"+s].shape==shape for s,shape in
       {"train":(4973,20,10),"val":(1065,20,10),"test":(1067,20,10)}.items()),
       "Expected 20-step, ten-feature shapes preserved.")
old_schema=json.loads((ORIGINAL/"feature_schema.json").read_text())
passed("ten_feature_schema_preserved",schema["ordered_features"]==old_schema["ordered_features"],
       "Feature names, order, formulas and units unchanged; implementation now matches definitions.")
passed("v1_artifacts_unchanged",original_artifacts_before=={name:sha256(ORIGINAL/name) for name in original_artifacts_before}
       and sha256(ORIGINAL/"catalog_artifact_hashes.json")==original_registry_hash,"Every registered v1 artifact and its hash registry unchanged.")
gap_ratios=manifest.input_to_target_gap_seconds.to_numpy()/old_manifest.input_to_target_gap_seconds.to_numpy()
impact={"raw_event_feature_changed_counts":raw_change_counts,"standardized_X_changed_cells":standardized_changes,
 "target_gap_corrected_to_original_ratio":{"min":float(gap_ratios.min()),"max":float(gap_ratios.max())},
 "rate_old_range":[float(old_raw[:,5].min()),float(old_raw[:,5].max())],
 "rate_corrected_range":[float(feature_array[:,5].min()),float(feature_array[:,5].max())],
 "dependency_start_changed_events":int((old_features.dependency_start_event_index.to_numpy()!=dependency_start).sum()),
 "left_censored_history_flag_changed_events":int((old_features.history_30d_left_censored.to_numpy()!=(ns-ns[0]<WINDOW_NS)).sum()),
 "scalers_refitted_from_scratch":True,"training_fit_events":len(train_event_idx),
 "new_rate_scaler_mean":float(scaler.mean_[5]),"new_rate_scaler_scale":float(scaler.scale_[5]),
 "source_csv_and_original_dataset_unchanged":True}
write_json(OUT/"timestamp_fix_impact.json",impact)

validation={"run_id":RUN_ID,"status":"APPROVED_RETROSPECTIVE_PREPROCESSING","training_ready":True,"timestamp_unit":"ns",
 "checks":checks,"splits":summary,"source_sha256":source_hash,
 "retained_records":len(catalog),"excluded_records":int((~keep).sum()),"warmup_events_without_targets":20,
 "raw_source_missing_counts":source_missing,"historical_context_overlap":overlap,
 "no_training":True,"no_optuna":True,"no_waveform_files_loaded":True}
write_json(OUT/"dataset_validation.json",validation)
artifact_names=["bilstm_catalog_seq20.pt","bilstm_catalog_seq20.npz","feature_schema.json",
 "target_definition.json","split_manifest.csv","preprocessing_config.json","catalog_train_scalers.json",
 "augmentation_config.json","dataset_validation.json","catalog_events.csv","catalog_feature_events.csv",
 "catalog_scaler_fit_events.csv","catalog_context_overlap.json","catalog_record_disposition.csv","catalog_magnitude_conflicts.csv",
 "timestamp_unit_tests.json","timestamp_fix_impact.json"]
write_json(OUT/"catalog_artifact_hashes.json",{"run_id":RUN_ID,"source_sha256":source_hash,
 "artifacts":{name:sha256(OUT/name) for name in artifact_names}})
npz.close()
print(json.dumps({"run_id":RUN_ID,"checks":{k:v["status"] for k,v in checks.items()},
                  "shapes":{s:summary[s]["X_shape"] for s in summary}},indent=2))


{
  "run_id": "945fdfac7b47cfb0",
  "checks": {
    "strict_input_before_target": "PASS",
    "correct_next_event_alignment": "PASS",
    "chronological_target_splits": "PASS",
    "unique_target_ids": "PASS",
    "past_only_feature_prefixes": "PASS",
    "train_only_fit_ids": "PASS",
    "train_only_fit_parameters": "PASS",
    "target_inverse_transform": "PASS",
    "numerical_artifact_equality": "PASS",
    "finite_model_arrays": "PASS",
    "saved_preprocessing_reconstruction": "PASS",
    "complete_event_provenance": "PASS",
    "augmentation_label_and_split_safety": "PASS",
    "source_csv_unchanged": "PASS",
    "operational_feature_availability": "UNKNOWN",
    "source_magnitude_calibration": "UNKNOWN",
    "duplicate_events_under_different_metadata": "UNKNOWN",
    "independent_30day_rates_all_events": "PASS",
    "independent_inter_event_days": "PASS",
    "independent_target_gap_seconds": "PASS",
    "known_timestamp_pairs_and_boundaries": "PASS",
    "original_split_and_ide

In [5]:
table=[]
for s in ("train","val","test"):
    d=summary[s]
    table.append(f"| {s} | {d['X_shape']} | {d['magnitude_mean']:.3f} | {100*d['M_ge_4_5_fraction']:.2f}% |")
report=f"""# Phase 1C: corrected CSV preprocessing
**APPROVED for retrospective research preprocessing. No training performed.** Run {RUN_ID}.

## Root cause and correction
Pandas parsed origin timestamps with microsecond resolution; native integer conversions were combined with nanosecond constants. Both feature-construction integer conversions now call utc_nanoseconds (explicit as_unit("ns")). WINDOW_NS, NS_PER_SECOND and SECONDS_PER_DAY define every conversion. No old tensors were multiplied to create new inputs.

The notebook recalculates inter-event days, 30-day counts/rates, target gaps, dependency intervals, initial-history flags and gap summaries from source timestamps. Rolling magnitude/depth summaries are also recomputed. New median imputation and StandardScaler are fitted from scratch on {len(train_event_idx)} unique training input events; no old fitted scaler is reused.

## Preserved design
Magnitude_Value with Magnitude_Type=ML; next retained ML event target and ML>=4.5 label. Twenty-step, ten-feature inputs. All event IDs, source timestamps, labels, split membership and cutoff records match v1. Historical overlap remains permitted and documented.

| Split | X shape | Target ML mean | ML>=4.5 |
|---|---|---:|---:|
{chr(10).join(table)}

## Independent verification and impact
{len(unit_results)} known-time tests passed: native microsecond/nanosecond inputs, sub-second and one-nanosecond intervals, timezone equivalence, leap day, sparse events and exact 30-day boundary exclusions. All retained-event rates agree with independent pandas time-index rolling counts. All inter-event days and target gaps agree with Python datetime differences.

Changed raw rate rows: {impact['raw_event_feature_changed_counts']['event_rate_30d']}/{len(catalog)}. Corrected rate range: {impact['rate_corrected_range']} events/day, versus {impact['rate_old_range']} previously. Target gaps are now {impact['target_gap_corrected_to_original_ratio']['min']:.0f}x the incorrectly stored values. Inter-event days were already correct and remain unchanged. Full per-feature/per-split changes: ../timestamp_fix_impact.json.

Strict input-before-target chronology, target-disjoint splits, training-only fitting, schema, shapes, finite arrays, inverse target transform, source magnitude preservation and exact PT/NPZ numerical equality passed. Original CSV and every registered v1 artifact retain their hashes. Critical failures raise assertions before approval.

## Limits and next stage
Reporting/revision times are unavailable: operational feature availability UNKNOWN. Source magnitude calibration and catalog completeness remain unverified. This is historical catalog context, not validated seconds-ahead EEW. Phase 1B must use this version exclusively and pass its own validation; no model training is included.

Arrays, parameters, manifests and validation reside in dataset/v2_timestamp_fixed/. Timestamp tests and impact details are versioned with the dataset. Notebook: phase1_bilstm_csv_preprocessing.ipynb.
"""
(REPORTS/"catalog_preprocessing_report.md").write_text(report,encoding="utf-8")
print("Dataset v2 APPROVED for retrospective preprocessing; augmentation revalidation is the next step.")


Dataset v2 APPROVED for retrospective preprocessing; augmentation revalidation is the next step.
